In [ ]:
import io
import textwrap
from dataclasses import dataclass
from pathlib import PurePath

import polars as pl

import pypdf

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch

FONT_SIZE = 11
ADDRESS_TOPLEFT_CORNER = [328, 688]
LINE_SIZE = 1.4
MAX_WIDTH = 35

In [ ]:
pl.Config.set_tbl_rows(20)

In [ ]:
def split_lines(adresse):
    return "\n".join(
        textwrap.fill(line, width=MAX_WIDTH) for line in adresse.split("\n")
    )

In [ ]:
@dataclass(frozen=True)
class Config:
    nom: str
    fichier: str
    f: int = 1
    m: int = 3

In [ ]:
SENT = {
    # ration 1
    "maires-parrains",
    "maires-corse",
    "maires-corse-parrains",
    # ration 2
    "cd-parrains",
    "cd-outremer-parrains",
    "cr-parrains",
    "cr-outremer-parrains",
    "membres-corse",
    "membres-guyane-parrains",
    "membres-martinique-parrains",
    # Ration 3
	"cd-outremer",
	"cr-outremer",
	"maires-nouvelle-caledonie",
	"maires-nouvelle-caledonie-parrains",
	"maires-outremer-parrains",
	"membres-corse-parrains",
	"membres-guyane",
	"membres-martinique",
	"membres-polynesie",
	"membres-saint-barthelemy",
	"membres-saint-martin",
	"membres-saint-pierre-et-miquelon",
	"membres-wallis-et-futuna",
}

In [ ]:
CONFIG_TEMPLATES = [
    Config("maires-parrains", "Templates/003_Courrier de JLM aux maires parrains de 2022_v4 (2).pdf"),
    Config("maires-corse-parrains", "Templates/004_Courrier_de_JLM_aux_maires_corses_parrains_de_2022_réélus.pdf"),
    Config("maires-nouvelle-caledonie-parrains", "Templates/005_Courrier_de_JLM_au_maire_de_Ouvéa_Nouvelle_Calédonie_parrain (3).pdf"),
    Config("cd-parrains", "Templates/006_Courrier de JLM parrains de 2022 CD, CR_ST.pdf"),
    Config("cd-outremer-parrains", "Templates/006_Courrier_de_JLM_parrains_de_2022_CD_et_CR_Reunion_et_Guadeloupe (2).pdf"),
    Config("cr-parrains", "Templates/006_Courrier de JLM parrains de 2022 CD, CR_ST.pdf", 5, 7),
    Config("cr-outremer-parrains", "Templates/006_Courrier_de_JLM_parrains_de_2022_CD_et_CR_Reunion_et_Guadeloupe (2).pdf", 5, 7),
    Config("maires-outremer-parrains", "Templates/015_Courrier maires ourte mer parrains 2022_ST.pdf"),
    Config("membres-guyane-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf"),
    Config("membres-martinique-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 5, 7),
    Config("membres-polynesie-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 9, 11),
    Config("membres-wallis-et-futuna-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 13, 15),
    Config("membres-saint-barthelemy-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 17, 19),
    Config("membres-saint-martin-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 21, 23),
    Config("membres-saint-pierre-et-miquelon-parrains", "Templates/006BIS_Courrier de JLM parrains de 2022 hors metropole_ST.pdf", 25, 27),
    Config("maires-corse", "Templates/008_Courrier aux maires, conseillers ass de Corse_ST.pdf"),
    Config("maires-nouvelle-caledonie", "Templates/009_Courrier aux maires de Nouvelle Calédonie_ST.pdf"),
    Config("membres-corse", "Templates/008_Courrier aux maires, conseillers ass de Corse_ST.pdf", f=5, m=7),
    Config("membres-corse-parrains", "Templates/008_Courrier aux conseillers ass de Corse parrain 22_v1.pdf"),
    Config("cr-outremer", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf"),
    Config("cd-outremer", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 5, 7),
    Config("membres-polynesie", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 9, 11),
    Config("membres-martinique", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 13, 15),
    Config("membres-guyane", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 17, 19),
    Config("membres-saint-martin", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 21, 23),
    Config("membres-saint-barthelemy", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 25, 27),
    Config("membres-saint-pierre-et-miquelon", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 29, 31),
    Config("membres-wallis-et-futuna", "Templates/017_Courrier membres des Assemblées d'outre mer_ST (2).pdf", 33, 35),
    Config("maires-outremer", "Templates/018_Courrier maires outre mer_ST.pdf"),
]
# catégories manquantes
# maires outremer
# cd / cr / assemblées outremer

In [ ]:
_templates = {
    path: pypdf.PdfReader(path) for path in {c.fichier for c in CONFIG_TEMPLATES}
}

pages = {
    c.nom: {
        "F": (_templates[c.fichier].get_page(c.f-1), _templates[c.fichier].get_page(c.f)),
        "M": (_templates[c.fichier].get_page(c.m-1), _templates[c.fichier].get_page(c.m)),
    }
    for c in CONFIG_TEMPLATES
}

In [ ]:
adresses_maires = pl.read_csv("out/adresses_maires_speciaux.csv")
adresses_territoriaux = pl.read_csv("out/adresses_territoriaux.csv")

adresses = pl.concat(
    [adresses_maires, adresses_territoriaux],
).with_columns(
    pl.col("adresse_complete").map_elements(split_lines)
)

In [ ]:
def overlay_adresse(adresse):
    overlay_buffer = io.BytesIO()
    cv = canvas.Canvas(overlay_buffer, pagesize=A4)

    font_size = FONT_SIZE
    lines = adresse.split("\n")

    if len(lines) >= 6:
        font_size -= 1
    
    cv.setFont(
        "Courier",
        font_size
    )

    for i, line in enumerate(lines):
        cv.drawString(
            ADDRESS_TOPLEFT_CORNER[0],
            ADDRESS_TOPLEFT_CORNER[1] - i * LINE_SIZE * font_size,
            line
        )

    cv.save()
    overlay_buffer.seek(0)
    
    return pypdf.PdfReader(overlay_buffer).get_page(0)

In [ ]:
def generer_page(writer, pages, sexe, adresse):
    page1 = writer.add_blank_page(*A4)
    page1.merge_page(pages[sexe][0])
    page1.merge_page(overlay_adresse(adresse))

    page2 = writer.add_blank_page(*A4)
    page2.merge_page(pages[sexe][1])

In [ ]:
adresses.filter(
    ~pl.col("courrier").is_in(pages)
)["courrier"].value_counts(sort=True)

In [ ]:
todo = adresses.filter(
    pl.col("courrier").is_in(pages)
    & ~pl.col("courrier").is_in(SENT)
).sort("courrier", "code", pl.col("nom_complet").str.extract(r" ([\w'-]+)$"))

todo["courrier"].value_counts(sort=True)

In [ ]:
todo.select(
    pl.col("adresse_complete").str.split("\n").list.len().max()
)

In [ ]:
writer = pypdf.PdfWriter()
for courrier, sexe, adresse in todo.select("courrier", "sexe", "adresse_complete").rows():
    generer_page(writer, pages[courrier], sexe, adresse)

writer.write("out/ration5.pdf")

for c in sorted(todo["courrier"].unique()):
    print(f'\t"{c}",')

## Générer courriers

In [ ]:
for courrier in pages:
    writer = pypdf.PdfWriter()
    for p in pages[courrier]["F"] + pages[courrier]["M"]:
        blank = writer.add_blank_page(*A4)
        blank.merge_page(p)
    writer.write(f"out/courrier/{courrier}.pdf")

In [ ]:
from collections import defaultdict

courriers_par_defaut = [
    "maires-parrains",
    "maires",
    "cd-parrains",
#    "cd",
    "cr-parrains",
#    "cr"
]

courriers = defaultdict(lambda: courriers_par_defaut)

courriers["20"] = [
    "maires-corse",
    "maires-corse-parrains",
    "membres-corse",
    "membres-corse-parrains"
]

courriers["971"] = courriers["972"] = [
    "maires-outremer",
    "maires-outremer-parrains",
    "cd-outremer",
    "cd-outremer-parrains",
    "cr-outremer",
    "cr-outremer-parrains",
]

courriers["972"] = [
    "maires-outremer",
    "membres-martinique",
    "membres-martinique-parrains",
]

courriers["973"] = [
    "maires-outremer",
    "membres-guyane",
    "membres-guyane-parrains",
]
courriers["975"] = [
    "membres-saint-pierre-et-miquelon"
]
courriers["976"] = [
    "maires-outremer",
    "maires-outremer-parrains",
    "cd-outremer",
    "cd-outremer-parrains"
]

courriers["977"] = [
    "membres-saint-barthelemy"
]
courriers["978"] = [
    "membres-saint-martin"
]

courriers["986"] = [
    "membres-wallis-et-futuna"
]

courriers["987"] = [
    "membres-polynesie",
    "membres-polynesie-parrains",
    "maires-outremer",
    "maires-outremer-parrains"
]

courriers["988"] = [
#    "membres-nouvelle-caledonie",
#    "membres-nouvelle-caledonie-parrains",
    "maires-nouvelle-caledonie",
    "maires-nouvelle-caledonie-parrains"
]

In [ ]:
from paramiko.client import SSHClient

client = SSHClient()
client.load_host_keys("/home/salome/Projets/playbooks/ssh_known_hosts")
client.connect("212.47.232.89", username="root")

sftp = client.open_sftp()

root = PurePath("/mnt/storage/data/__groupfolders/8/files")

for f in Path("out/extraction").glob("*.xlsx"):
    dep = f.stem

    cs = courriers[dep]

    for c in cs:
        sftp.put(f"out/courrier/{c}.pdf", str(root / dep / f"{c}.pdf"))

In [ ]:
base_url = "https://cloud.franceinsoumise.org"
ocs = f"{base_url}/ocs/v2.php/apps/files_sharing/api/v1"

In [ ]:
import webbrowser

import niquests
from niquests.auth import BearerTokenAuth
nc_info = niquests.post(f"{base_url}/index.php/login/v2").json()

webbrowser.open(nc_info["login"])

nc_info

In [ ]:
res = niquests.post(
  nc_info["poll"]["endpoint"],
  data={"token": nc_info["poll"]["token"]},
)

token = res.json()["appPassword"]

In [ ]:
session = niquests.Session()
session.auth = BearerTokenAuth(token)
session.headers["OCS-APIRequest"] = "true"

In [ ]:
def share(path):
    params = {
        "path": path,
        "shareType": 3,
        "permissions": 1,
        "expireDate": "2027-05-31",
    }

    res = session.post(f"{ocs}/shares", data=params)

    if res.status_code != 200:
        raise RuntimeError(f"Requête incorrecte : {res.text}", res)

    tree = etree.fromstring(res.text)
    url = tree.xpath("/ocs/data/url")[0].text
    return url

In [ ]:
liens = {}
for f in Path("out/extraction").glob("*.xlsx"):
    dep = f.stem
    if dep in liens:
        continue

    liens[dep] = share(f"/Parrainages 2027/{dep}")

In [ ]:
df = pl.DataFrame({"code_departement": liens.keys(), "lien": liens.values()}).sort("code_departement")

In [ ]:
df.write_excel("out/liste_liens.xlsx")